# Representação e descritores de regiões

Este notebook transforma regiões já segmentadas em fronteiras, medidas geométricas e momentos básicos.

**Pré-requisitos:** máscaras binárias, componentes conectados e coordenadas de imagens.

## Objetivos

Extrair contornos, calcular área, perímetro, centroide, bounding box, razão de aspecto, circularidade e momentos. A segmentação é uma etapa separada: os descritores recebem regiões já segmentadas.

## 1. Instalação

In [ ]:
%pip install -q "git+https://github.com/tfvieira/dip-2026-2.git"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from dip_toolkit.modules.image_segmenter import ImageSegmenter
from dip_toolkit.modules.region_descriptor import RegionDescriptorExtractor

segmenter = ImageSegmenter()
extractor = RegionDescriptorExtractor()

## 2. Regiões segmentadas

A máscara usa `0` para fundo e `255` para primeiro plano. `ImageSegmenter.connected_components` produz IDs consecutivos; o fundo recebe o rótulo `0`.

In [ ]:
mask = np.zeros((100, 140), dtype=np.uint8)
mask[15:45, 20:55] = 255
mask[55:85, 80:125] = 255
regions = segmenter.connected_components(mask, connectivity=8)

figure, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(regions.mask, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Máscara binária")
axes[1].imshow(regions.labels, cmap="nipy_spectral")
axes[1].set_title("Mapa de rótulos")
for axis in axes:
    axis.axis("off")
plt.show()

## 3. Descritores por região

A área reportada é a contagem de pixels. O centroide vem dos momentos brutos. A circularidade é calculada como `4 pi contour_area / perimeter²`, usando a área geométrica e o perímetro do contorno, e vale `0` quando o contorno é degenerado.

In [ ]:
descriptors = extractor.describe_regions(regions)
for descriptor in descriptors:
    print(
        f"ID={descriptor.region_id}, área={descriptor.area:.0f}, "
        f"centroide={descriptor.centroid}, caixa={descriptor.bounding_box}"
    )
    print(f"  momentos: {descriptor.moments}")

## 4. Visualização separada

A visualização não altera as medidas: ela apenas desenha contornos, caixas e centroides sobre a máscara com Matplotlib.

In [ ]:
figure, axis = extractor.plot_regions(regions.mask, descriptors)
plt.show()

## Exercício

Crie uma terceira região e observe como ID, centroide, caixa e momentos mudam. Depois compare objetos que se tocam apenas na diagonal usando conectividade 4 e 8.